In [1]:
import pandas as pd
import csv
import pandas_gbq
import time
from datetime import datetime
import os
from google.oauth2 import service_account
from google.cloud import bigquery
from datetime import datetime
from dateutil.relativedelta import relativedelta
import numpy as np

In [2]:
CREDS = '../../converge-database-0331482f2ee5.json'
client = bigquery.Client.from_service_account_json(json_credentials_path=CREDS)

In [3]:
combined = pd.DataFrame()
set_month = "202601"

In [9]:
combined = pd.read_excel('I:New Structure/Actuarial New/Database/Farmers/'+set_month+'/01 26 Converge MYGA Revised.xlsx', sheet_name='Combined GL')

In [304]:
av_seriatim = pd.read_csv('I:New Structure/Actuarial New/Database/Farmers/'+set_month+'/03 26 Converge AV RF.csv')

In [10]:
combined.columns = combined.columns.map(str.lower)
combined.columns = combined.columns.map(lambda x : x.replace(" " , "_"))

In [11]:
combined

,journal_entry,series,trx_date,account_number,account_description,debit_amount,credit_amount,net,description,reference,...,converge_credit,converge_net,policy_plan,policy_term,issue_date,ceding_allowance,unnamed:_20,unnamed:_21,pre-8/1,post-8/1
0,7344,Purchasing,2026-01-05,1-C00-5100-101,Surrender Benefits Paid-MYGA,603.67,0.00,603.67,Annuity Regular Distribution,Annuity Regular Distribution,...,0.000,90.5505,ASPQ05,5,2022-10-25,0.0,NaN,3.0,0.0,0.015
1,7347,Purchasing,2026-01-05,1-C00-5100-101,Surrender Benefits Paid-MYGA,243.80,0.00,243.80,Annuity Regular Distribution,Annuity Regular Distribution,...,0.000,36.5700,ASPN05,5,2023-03-27,0.0,NaN,5.0,0.0,0.015
2,7348,Purchasing,2026-01-05,1-C00-5100-101,Surrender Benefits Paid-MYGA,231.91,0.00,231.91,Annuity Regular Distribution,Annuity Regular Distribution,...,0.000,69.5730,ASPN03,3,2023-08-07,0.0,NaN,7.0,0.0,0.015
3,7351,Purchasing,2026-01-05,1-C00-5100-101,Surrender Benefits Paid-MYGA,231.28,0.00,231.28,Annuity Regular Distribution,Annuity Regular Distribution,...,0.000,34.6920,ASPN05,5,2023-07-28,0.0,NaN,10.0,0.0,0.015
4,7352,Purchasing,2026-01-05,1-C00-5100-101,Surrender Benefits Paid-MYGA,1446.43,0.00,1446.43,Annuity Regular Distribution,Annuity Regular Distribution,...,0.000,578.5720,ASPN05,5,2024-01-29,0.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1194,7733,Financial,2026-01-28,1-C00-6000-101,Commissions,0.00,426.42,-426.42,Commissions Expense-MYGA,18795F,...,341.136,-341.1360,AMPQ07,7,2024-12-02,0.0,NaN,NaN,NaN,NaN
1195,7733,Financial,2026-01-28,1-C00-6000-101,Commissions,0.00,213.21,-213.21,Commissions Expense-MYGA,18795F,...,170.568,-170.5680,AMPQ07,7,2024-12-02,0.0,NaN,NaN,NaN,NaN
1196,7733,Financial,2026-01-28,1-C00-6000-101,Commissions,0.00,537.84,-537.84,Commissions Expense-MYGA,18524F,...,430.272,-430.2720,AMPN05,5,2024-12-03,0.0,NaN,NaN,NaN,NaN
1197,7733,Financial,2026-01-28,1-C00-6000-101,Commissions,0.00,94.91,-94.91,Commissions Expense-MYGA,18524F,...,75.928,-75.9280,AMPN05,5,2024-12-03,0.0,NaN,NaN,NaN,NaN


In [12]:
combined.debit_amount.replace(" -   ", np.nan, inplace=True)
combined.credit_amount.replace(" -   ", np.nan, inplace=True)

In [13]:
combined = combined.astype({"trx_date" : "datetime64[ns]", "debit_amount" : "float64", "credit_amount" : "float64" ,"originating_master_id" :"str", "converge_quota_share" : "float64","net" : "float64"})


In [14]:
combined = combined.iloc[:, 0:19]
combined['set_month'] = set_month

In [15]:
distribution_types = {
    'Annuity Full Surrender': 'full_surrender',
    'Annuity Partial Surrender': 'partial_surrender',
    'Annuity RMD': 'rmd',
    'Annuity Regular Distribution': 'other'
}

In [16]:
for desc, table_name in distribution_types.items():

    df = combined[
        (combined['account_number'] == '1-C00-5100-101') &
        (combined['description'] == desc)
    ].copy()

    df.rename(columns={
        "policy_id": "policy_number",
        "converge_quota_share": "quota_share",
        "net": "av_withdrawn",
        "policy_plan": "plan"
    }, inplace=True)

    df = df[[
        "policy_number",
        "plan",
        "av_withdrawn",
        "quota_share"
    ]]
    df["set_month"] = set_month
    df.info()
    try:
        df.to_gbq(
            destination_table=f"farmers.{table_name}",
            project_id="converge-database",
            if_exists="append"
        )

        print(f"✅ Uploaded {desc} ({len(df)} rows) -> farmers.{table_name}")

    except Exception as e:
        print(f"❌ Failed to upload {desc}: {e}")

<class 'pandas.core.frame.DataFrame'>
Int64Index: 5 entries, 48 to 71
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   policy_number  5 non-null      object 
 1   plan           5 non-null      object 
 2   av_withdrawn   5 non-null      float64
 3   quota_share    5 non-null      float64
 4   set_month      5 non-null      object 
dtypes: float64(2), object(3)
memory usage: 240.0+ bytes


100%|██████████| 1/1 [00:00<?, ?it/s]


✅ Uploaded Annuity Full Surrender (5 rows) -> farmers.full_surrender
<class 'pandas.core.frame.DataFrame'>
Int64Index: 5 entries, 61 to 97
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   policy_number  5 non-null      object 
 1   plan           5 non-null      object 
 2   av_withdrawn   5 non-null      float64
 3   quota_share    5 non-null      float64
 4   set_month      5 non-null      object 
dtypes: float64(2), object(3)
memory usage: 240.0+ bytes


100%|██████████| 1/1 [00:00<?, ?it/s]


✅ Uploaded Annuity Partial Surrender (5 rows) -> farmers.partial_surrender
<class 'pandas.core.frame.DataFrame'>
Int64Index: 25 entries, 11 to 117
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   policy_number  25 non-null     object 
 1   plan           25 non-null     object 
 2   av_withdrawn   25 non-null     float64
 3   quota_share    25 non-null     float64
 4   set_month      25 non-null     object 
dtypes: float64(2), object(3)
memory usage: 1.2+ KB


100%|██████████| 1/1 [00:00<?, ?it/s]


✅ Uploaded Annuity RMD (25 rows) -> farmers.rmd
<class 'pandas.core.frame.DataFrame'>
Int64Index: 71 entries, 0 to 122
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   policy_number  71 non-null     object 
 1   plan           71 non-null     object 
 2   av_withdrawn   71 non-null     float64
 3   quota_share    71 non-null     float64
 4   set_month      71 non-null     object 
dtypes: float64(2), object(3)
memory usage: 3.3+ KB


100%|██████████| 1/1 [00:00<?, ?it/s]

✅ Uploaded Annuity Regular Distribution (71 rows) -> farmers.other


In [17]:
premium = combined[
    (combined['account_number'].isin([
        '1-C00-4000-101',
        '1-C00-4001-101'
    ])) &
    (combined['converge_quota_share'] > 0)
]

In [18]:
premium

,journal_entry,series,trx_date,account_number,account_description,debit_amount,credit_amount,net,description,reference,originating_master_id,policy_id,converge_quota_share,converge_debit,converge_credit,converge_net,policy_plan,policy_term,issue_date,set_month
881,7733,Financial,2026-01-15,1-C00-4001-101,Premium Income,0.0,25306.01,-25306.01,Annuity Prems-MYGA-Qual,R13458F,1152601,R13458F,0.4,0.0,10122.404,-10122.404,ASPQ05,5,2024-01-08,202601


In [19]:
premium = premium.rename(columns = {"net" : "gross_premium", "converge_quota_share" : "quota_share", "policy_id" : "policy_number"})
premium['set_month'] = set_month

In [20]:
premium = premium[['policy_number', 'gross_premium', 'quota_share', 'set_month']]

In [21]:
premium.to_gbq("converge-database.farmers.premium",
                  if_exists='append',
                  project_id="converge-database")

100%|██████████| 1/1 [00:00<?, ?it/s]


In [22]:
death = combined[
        (combined['account_number'] == '1-C00-5000-101') &
        (combined['converge_quota_share'] > 0)
    ]

In [25]:
death

,journal_entry,series,trx_date,account_number,account_description,debit_amount,credit_amount,net,description,reference,originating_master_id,policy_id,converge_quota_share,converge_debit,converge_credit,converge_net,policy_plan,policy_term,issue_date,set_month
5,7353,Purchasing,2026-01-05,1-C00-5000-101,Death Benefits: Paid-MYGA,37890.85,0.0,37890.85,Death Claim,Death Claim,P13716F -001,13716F,0.40,15156.3400,0.0,15156.3400,ASPN03,3,2024-02-12,202601
18,7366,Purchasing,2026-01-05,1-C00-5000-101,Death Benefits: Paid-MYGA,82398.75,0.0,82398.75,Death Claim,Death Claim,P16379F -001,16379F,0.40,32959.5000,0.0,32959.5000,AMPN05,5,2024-08-05,202601
20,7369,Purchasing,2026-01-05,1-C00-5000-101,Death Benefits: Paid-MYGA,25201.47,0.0,25201.47,Death Claim,Death Claim,P18947F -001,18947F,0.80,20161.1760,0.0,20161.1760,AMPQ03,3,2024-12-17,202601
21,7370,Purchasing,2026-01-05,1-C00-5000-101,Death Benefits: Paid-MYGA,57434.70,0.0,57434.70,Death Claim,Death Claim,P18955F -001,18955F,0.80,45947.7600,0.0,45947.7600,AMPN05,5,2024-12-03,202601
53,7470,Purchasing,2026-01-12,1-C00-5000-101,Death Benefits: Paid-MYGA,222569.19,0.0,222569.19,Death Claim,Death Claim,P13782F -001,13782F,0.40,89027.6760,0.0,89027.6760,ASPN05,5,2024-02-05,202601
54,7471,Purchasing,2026-01-12,1-C00-5000-101,Death Benefits: Paid-MYGA,12396.38,0.0,12396.38,Death Claim,Death Claim,P13927F -001,13927F,0.40,4958.5520,0.0,4958.5520,ASPQ05,5,2024-02-23,202601
55,7472,Purchasing,2026-01-12,1-C00-5000-101,Death Benefits: Paid-MYGA,213212.09,0.0,213212.09,Death Claim,Death Claim,P17069F -001,17069F,0.80,170569.6720,0.0,170569.6720,AMPN05,5,2024-10-07,202601
56,7478,Purchasing,2026-01-12,1-C00-5000-101,Death Benefits: Paid-MYGA,84809.39,0.0,84809.39,Death Claim,Death Claim,P22514F -001,22514F,0.10,8480.9390,0.0,8480.9390,AMPN05,5,2025-06-09,202601
91,7584,Purchasing,2026-01-20,1-C00-5000-101,Death Benefits: Paid-MYGA,139060.93,0.0,139060.93,Death Claim,Death Claim,P11202F -001,11202F,0.15,20859.1395,0.0,20859.1395,ASPN05,5,2023-04-20,202601
100,7594,Purchasing,2026-01-20,1-C00-5000-101,Death Benefits: Paid-MYGA,276750.13,0.0,276750.13,Death Claim,Death Claim,P13889F -001,13889F,0.40,110700.0520,0.0,110700.0520,ASPN10,10,2024-02-26,202601


In [27]:
death.net_amount.sum()

2290721.3

In [26]:
death = death.rename(columns = {"net" : "net_amount", "converge_quota_share" : "quotashare", "policy_id" : "policy_number", "issue_date" : "missuedt", "policy_plan" : "plan"})
death['set_month']=set_month

In [28]:
death = death[["policy_number", "net_amount", "missuedt", "plan", "quotashare", "set_month"]]

In [29]:
death.to_gbq("converge-database.farmers.death_claims",
                  if_exists='append',
                  project_id="converge-database")

100%|██████████| 1/1 [00:00<?, ?it/s]


In [30]:
combined = combined.iloc[:, 0:16]

In [31]:
combined['set_month'] = set_month
combined = combined.rename(columns = {"net" : "diff", "converge_quota_share" : "quota_share"})

In [32]:
combined = combined.drop(['converge_quota_share', 'converge_debit', 'converge_net', 'converge_credit'], axis=1, errors="ignore")

In [33]:
combined

,journal_entry,series,trx_date,account_number,account_description,debit_amount,credit_amount,diff,description,reference,originating_master_id,policy_id,quota_share,set_month
0,7344,Purchasing,2026-01-05,1-C00-5100-101,Surrender Benefits Paid-MYGA,603.67,0.00,603.67,Annuity Regular Distribution,Annuity Regular Distribution,P10409F -001,10409F,0.15,202601
1,7347,Purchasing,2026-01-05,1-C00-5100-101,Surrender Benefits Paid-MYGA,243.80,0.00,243.80,Annuity Regular Distribution,Annuity Regular Distribution,P10744F -001,10744F,0.15,202601
2,7348,Purchasing,2026-01-05,1-C00-5100-101,Surrender Benefits Paid-MYGA,231.91,0.00,231.91,Annuity Regular Distribution,Annuity Regular Distribution,P11319F -001,11319F,0.30,202601
3,7351,Purchasing,2026-01-05,1-C00-5100-101,Surrender Benefits Paid-MYGA,231.28,0.00,231.28,Annuity Regular Distribution,Annuity Regular Distribution,P11868F -001,11868F,0.15,202601
4,7352,Purchasing,2026-01-05,1-C00-5100-101,Surrender Benefits Paid-MYGA,1446.43,0.00,1446.43,Annuity Regular Distribution,Annuity Regular Distribution,P13638F -001,13638F,0.40,202601
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1194,7733,Financial,2026-01-28,1-C00-6000-101,Commissions,0.00,426.42,-426.42,Commissions Expense-MYGA,18795F,1282601,18795F,0.80,202601
1195,7733,Financial,2026-01-28,1-C00-6000-101,Commissions,0.00,213.21,-213.21,Commissions Expense-MYGA,18795F,1282601,18795F,0.80,202601
1196,7733,Financial,2026-01-28,1-C00-6000-101,Commissions,0.00,537.84,-537.84,Commissions Expense-MYGA,18524F,1282601,18524F,0.80,202601
1197,7733,Financial,2026-01-28,1-C00-6000-101,Commissions,0.00,94.91,-94.91,Commissions Expense-MYGA,18524F,1282601,18524F,0.80,202601


In [34]:
av_seriatim

NameError: name 'av_seriatim' is not defined

In [306]:
av_seriatim.columns = av_seriatim.columns.map(str.lower)
av_seriatim.columns = av_seriatim.columns.map(lambda x : x.replace(" " , "_"))
av_seriatim.columns = av_seriatim.columns.map(lambda x : x.replace("-" , "_"))

In [308]:
av_seriatim = av_seriatim.dropna(subset=["policy"]).reset_index(drop=True)

In [35]:
combined.to_gbq("converge-database.farmers.combined_gl",
                  if_exists='append',
                  project_id="converge-database")

100%|██████████| 1/1 [00:00<?, ?it/s]


In [36]:
import sys

# Add directory containing LDTI.py
sys.path.append('../../actuarial-pipelines/reconciliations/farmers/')

# Import the function
from reconciliation import run_reconciliation

# Trigger the AVRF analysis
run_reconciliation(set_month, "farmers")


    SELECT SUM(net_amount) AS total
    FROM `farmers.death_claims`
    WHERE set_month = '202601'
    
{'product': 'farmers', 'fieldname': 'death_claims', 'total': 2290721.3000000003}
-----------------------------------------------

    SELECT SUM(net_amount) AS total
    FROM `farmers.cancellation`
    WHERE set_month = '202601'
    
{'product': 'farmers', 'fieldname': 'cancellation', 'total': None}
-----------------------------------------------

    select sum(av_withdrawn*quota_share) FROM `farmers.full_surrender` WHERE set_month = "202601"
    
{'product': 'farmers', 'fieldname': 'full_surrender', 'total': 199005.82650000002}
-----------------------------------------------

    select sum(gross_premium*quota_share) FROM `farmers.premium` WHERE set_month = "202601"
    
{'product': 'farmers', 'fieldname': 'premium', 'total': -10122.404}
-----------------------------------------------

    select sum(av_withdrawn*quota_share) FROM `farmers.partial_surrender` WHERE set_month = "202